# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/124pritivarma6001-commits/flyrank_internship_ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

The baseline rule identifies content that has good search visibility but is receiving relatively few clicks.

A content item is classified as an **OPPORTUNITY** when:
- Impressions are at least 237
- CTR is below 0.003026
- Average position is 20 or better

The corresponding reason code is:

`LOW_CTR_GOOD_POSITION`

This reason code explains that the content has a relatively low CTR despite having a good search position, indicating a potential opportunity for content improvement or refresh.

In [20]:
%pip -q install duckdb huggingface_hub

In [21]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [22]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [23]:
ctr_distribution = con.sql(f"""
WITH content_level AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(
            CASE
                WHEN gsc_impressions > 0
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        ) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
    FROM {TABLES['fact_daily']}
    WHERE gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
      AND gsc_avg_position IS NOT NULL
    GROUP BY content_hash_id
),
base AS (
    SELECT
        *,
        CAST(clicks AS DOUBLE) / NULLIF(impressions, 0) AS ctr
    FROM content_level
    WHERE impressions > 0
)
SELECT
    COUNT(*) AS total_contents,
    COUNT(*) FILTER (WHERE clicks > 0) AS contents_with_clicks,
    COUNT(*) FILTER (WHERE clicks = 0) AS contents_with_zero_clicks,
    MIN(ctr) AS min_ctr,
    MAX(ctr) AS max_ctr,
    QUANTILE_CONT(ctr, 0.25) AS ctr_25th,
    QUANTILE_CONT(ctr, 0.50) AS ctr_median,
    QUANTILE_CONT(ctr, 0.75) AS ctr_75th
FROM base
""").df()

ctr_distribution

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_contents,contents_with_clicks,contents_with_zero_clicks,min_ctr,max_ctr,ctr_25th,ctr_median,ctr_75th
0,309234,148941,160293,0.0,1.0,0.0,0.0,0.003024


In [24]:
ctr_threshold = con.sql(f"""
WITH content_level AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN CAST(SUM(gsc_clicks) AS DOUBLE) / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr
    FROM {TABLES['fact_daily']}
    WHERE gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
)
SELECT
    QUANTILE_CONT(ctr, 0.75) AS ctr_threshold,
    QUANTILE_CONT(impressions, 0.50) AS impression_threshold
FROM content_level
WHERE ctr IS NOT NULL
""").df()

ctr_threshold

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ctr_threshold,impression_threshold
0,0.003026,237.0


In [25]:
rule_preview = con.sql(f"""
WITH content_level AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN CAST(SUM(gsc_clicks) AS DOUBLE) / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr,
        SUM(gsc_avg_position * gsc_impressions)
        / NULLIF(SUM(gsc_impressions), 0) AS avg_position

    FROM {TABLES['fact_daily']}

    WHERE gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
      AND gsc_avg_position IS NOT NULL

    GROUP BY content_hash_id

    HAVING SUM(gsc_impressions) > 0
),

scored AS (
    SELECT
        *,
        CASE
            WHEN impressions >= 237
             AND ctr < 0.003026
             AND avg_position <= 20
            THEN 'OPPORTUNITY'
            ELSE 'NOT OPPORTUNITY'
        END AS rule_result,

        CASE
            WHEN impressions >= 237
             AND ctr < 0.003026
             AND avg_position <= 20
            THEN 'LOW_CTR_GOOD_POSITION'
            ELSE 'NOT_OPPORTUNITY'
        END AS reason_code

    FROM content_level
)

SELECT
    rule_result,
    reason_code,
    COUNT(*) AS rows_count
FROM scored
GROUP BY rule_result, reason_code
ORDER BY rule_result, reason_code
""").df()

rule_preview

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rule_result,reason_code,rows_count
0,NOT OPPORTUNITY,NOT_OPPORTUNITY,242472
1,OPPORTUNITY,LOW_CTR_GOOD_POSITION,66762


## 2. Build the ranked queue (writes the CSV)

I assign a baseline action score to each content item using the rule result,ctr,impressions, and average position. Opportunity rows receive higher scores so they appear first in the review queue. Each row also gets a reason code and action label. The queue is saved as 'work/outputs/baseline_action_score.csv'

In [26]:
import os

# Create output folder if it does not exist
os.makedirs("work/outputs", exist_ok=True)

ranked_queue = con.sql(f"""
WITH content_level AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN CAST(SUM(gsc_clicks) AS DOUBLE) / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(
                CASE
                    WHEN gsc_impressions > 0
                    THEN gsc_avg_position * gsc_impressions
                    ELSE 0
                END
            ) / SUM(gsc_impressions)
            ELSE NULL
        END AS avg_position

    FROM {TABLES['fact_daily']}

    WHERE gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
      AND gsc_avg_position IS NOT NULL

    GROUP BY content_hash_id

    HAVING SUM(gsc_impressions) > 0
),

scored AS (
    SELECT
        *,

        CASE
            WHEN impressions >= 237
                 AND ctr < 0.003026
                 AND avg_position <= 20
            THEN 'OPPORTUNITY'
            ELSE 'NOT OPPORTUNITY'
        END AS rule_result,

        CASE
            WHEN impressions >= 237
                 AND ctr < 0.003026
                 AND avg_position <= 20
            THEN 'LOW_CTR_GOOD_POSITION'
            ELSE 'NOT_OPPORTUNITY'
        END AS reason_code

    FROM content_level
),

ranked AS (
    SELECT
        content_hash_id,
        impressions,
        clicks,
        ctr,
        avg_position,
        rule_result,
        reason_code,

        CASE
            WHEN rule_result = 'OPPORTUNITY'
            THEN
                100
                * (0.003026 - ctr) / 0.003026
                + (20 - avg_position) / 20.0 * 10
            ELSE 0
        END AS action_score,

        CASE
            WHEN rule_result = 'OPPORTUNITY'
            THEN 'REFRESH_CONTENT'
            ELSE 'NO_ACTION'
        END AS action

    FROM scored
)

SELECT
    *,
    ROW_NUMBER() OVER (
        ORDER BY action_score DESC
    ) AS rank

FROM ranked
ORDER BY action_score DESC
""").df()

# Save ranked queue
output_path = "work/outputs/baseline_action_score.csv"
ranked_queue.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows written: {len(ranked_queue):,}")

# Show top 10
ranked_queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: work/outputs/baseline_action_score.csv
Rows written: 309,234


,content_hash_id,impressions,clicks,ctr,avg_position,rule_result,reason_code,action_score,action,rank
0,content_b3f11bfce6523f8f,659.0,0.0,0.0,0.324734,OPPORTUNITY,LOW_CTR_GOOD_POSITION,109.837633,REFRESH_CONTENT,1
1,content_85fbeb8bf5fbb90e,2497.0,0.0,0.0,0.340008,OPPORTUNITY,LOW_CTR_GOOD_POSITION,109.829996,REFRESH_CONTENT,2
2,content_7ea6d35866735527,4094.0,0.0,0.0,0.343185,OPPORTUNITY,LOW_CTR_GOOD_POSITION,109.828407,REFRESH_CONTENT,3
3,content_eac0a973f63c8b0d,272.0,0.0,0.0,0.437500,OPPORTUNITY,LOW_CTR_GOOD_POSITION,109.781250,REFRESH_CONTENT,4
4,content_4aaa40c22653168a,1496.0,0.0,0.0,0.443182,OPPORTUNITY,LOW_CTR_GOOD_POSITION,109.778409,REFRESH_CONTENT,5
5,content_73ddaea083e67035,4363.0,0.0,0.0,0.445107,OPPORTUNITY,LOW_CTR_GOOD_POSITION,109.777447,REFRESH_CONTENT,6
6,content_7d385fd64452fd29,4107.0,0.0,0.0,0.448503,OPPORTUNITY,LOW_CTR_GOOD_POSITION,109.775749,REFRESH_CONTENT,7
7,content_bc70bdc12eb6657c,9222.0,0.0,0.0,0.486554,OPPORTUNITY,LOW_CTR_GOOD_POSITION,109.756723,REFRESH_CONTENT,8
8,content_41f5aaee76079b6b,261.0,0.0,0.0,0.513410,OPPORTUNITY,LOW_CTR_GOOD_POSITION,109.743295,REFRESH_CONTENT,9
9,content_fb352195039d88b3,1293.0,0.0,0.0,0.530549,OPPORTUNITY,LOW_CTR_GOOD_POSITION,109.734725,REFRESH_CONTENT,10


## 3. Top-20 review


I reviewed the top 20 ranked opportunities from the baseline action score.
The selected rows are flagged for refresh because they have low CTR while maintaining a good average search position.

The confidence note is rule-based, not a model probability. A pick could be wrong if the low CTR is caused by search intent, seasonal behavior, insufficient data quality, or other factors not captured by the baseline rule.

In [27]:
# Section 3: Top-20 review

top20_review = ranked_queue.head(20).copy()

top20_review["confidence_note"] = top20_review.apply(
    lambda row: (
        "High rule confidence: low CTR with good position and sufficient impressions."
        if row["rule_result"] == "OPPORTUNITY"
        else "Low confidence: does not meet the opportunity rule."
    ),
    axis=1
)

top20_review["what_would_make_it_wrong"] = (
    "Low CTR may be caused by search intent, seasonality, data quality, "
    "or factors not captured by the baseline rule."
)

top20_review = top20_review[
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "action_score",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

top20_review

,rank,content_hash_id,action,reason_code,action_score,confidence_note,what_would_make_it_wrong
0,1,content_b3f11bfce6523f8f,REFRESH_CONTENT,LOW_CTR_GOOD_POSITION,109.837633,High rule confidence: low CTR with good positi...,"Low CTR may be caused by search intent, season..."
1,2,content_85fbeb8bf5fbb90e,REFRESH_CONTENT,LOW_CTR_GOOD_POSITION,109.829996,High rule confidence: low CTR with good positi...,"Low CTR may be caused by search intent, season..."
2,3,content_7ea6d35866735527,REFRESH_CONTENT,LOW_CTR_GOOD_POSITION,109.828407,High rule confidence: low CTR with good positi...,"Low CTR may be caused by search intent, season..."
3,4,content_eac0a973f63c8b0d,REFRESH_CONTENT,LOW_CTR_GOOD_POSITION,109.781250,High rule confidence: low CTR with good positi...,"Low CTR may be caused by search intent, season..."
4,5,content_4aaa40c22653168a,REFRESH_CONTENT,LOW_CTR_GOOD_POSITION,109.778409,High rule confidence: low CTR with good positi...,"Low CTR may be caused by search intent, season..."
5,6,content_73ddaea083e67035,REFRESH_CONTENT,LOW_CTR_GOOD_POSITION,109.777447,High rule confidence: low CTR with good positi...,"Low CTR may be caused by search intent, season..."
6,7,content_7d385fd64452fd29,REFRESH_CONTENT,LOW_CTR_GOOD_POSITION,109.775749,High rule confidence: low CTR with good positi...,"Low CTR may be caused by search intent, season..."
7,8,content_bc70bdc12eb6657c,REFRESH_CONTENT,LOW_CTR_GOOD_POSITION,109.756723,High rule confidence: low CTR with good positi...,"Low CTR may be caused by search intent, season..."
8,9,content_41f5aaee76079b6b,REFRESH_CONTENT,LOW_CTR_GOOD_POSITION,109.743295,High rule confidence: low CTR with good positi...,"Low CTR may be caused by search intent, season..."
9,10,content_fb352195039d88b3,REFRESH_CONTENT,LOW_CTR_GOOD_POSITION,109.734725,High rule confidence: low CTR with good positi...,"Low CTR may be caused by search intent, season..."


## 4. Weak picks + leakage check


I reviewed the baseline-ranked opportunities for weak or questionable picks.
The main weak-pick risk is that a low CTR may not always indicate a content problem.

I also checked the rule inputs for leakage. The baseline uses current impressions, CTR, and average search position only. No future-window information or product flags are used in the scoring rule.

In [28]:
# Section 4: Weak picks + leakage check

# Check the lowest-ranked opportunities
weak_picks = ranked_queue[
    ranked_queue["rule_result"] == "OPPORTUNITY"
].sort_values("action_score").head(10)

print("Weakest 10 opportunity picks:")
display(
    weak_picks[
        [
            "content_hash_id",
            "impressions",
            "clicks",
            "ctr",
            "avg_position",
            "action_score",
            "reason_code",
            "action"
        ]
    ]
)

print("\nLeakage check:")

print("Future-window fields used in rule: NONE")
print("Product flags used in rule: NONE")
print("Rule inputs: impressions, CTR, average position")

print("\nConclusion:")
print(
    "No future-window or product-flag leakage was used in the baseline scoring rule."
)

Weakest 10 opportunity picks:


,content_hash_id,impressions,clicks,ctr,avg_position,action_score,reason_code,action
66761,content_894e708aa6a2b555,53922.0,163.0,0.003023,19.748322,0.228783,LOW_CTR_GOOD_POSITION,REFRESH_CONTENT
66760,content_2376a2fccbdcb94e,661.0,2.0,0.003026,19.180030,0.419284,LOW_CTR_GOOD_POSITION,REFRESH_CONTENT
66759,content_9942cb40b3a0e089,30789.0,93.0,0.003021,19.067362,0.646118,LOW_CTR_GOOD_POSITION,REFRESH_CONTENT
66758,content_f9b8c01aa70ddd42,7652.0,23.0,0.003006,19.873889,0.732251,LOW_CTR_GOOD_POSITION,REFRESH_CONTENT
66757,content_0d40b092655b276a,180734.0,545.0,0.003015,19.164667,0.765277,LOW_CTR_GOOD_POSITION,REFRESH_CONTENT
66756,content_64cc412654de499d,5295.0,16.0,0.003022,18.734466,0.774254,LOW_CTR_GOOD_POSITION,REFRESH_CONTENT
66755,content_2377152bad7dfec1,333.0,1.0,0.003003,19.723724,0.898118,LOW_CTR_GOOD_POSITION,REFRESH_CONTENT
66754,content_0de9ae000b6c69ce,12342.0,37.0,0.002998,19.930400,0.963638,LOW_CTR_GOOD_POSITION,REFRESH_CONTENT
66753,content_5f93f846d9834caf,1329.0,4.0,0.003010,18.969150,1.051387,LOW_CTR_GOOD_POSITION,REFRESH_CONTENT
66752,content_2716b9559eae2ddd,6648.0,20.0,0.003008,18.903279,1.129207,LOW_CTR_GOOD_POSITION,REFRESH_CONTENT



Leakage check:
Future-window fields used in rule: NONE
Product flags used in rule: NONE
Rule inputs: impressions, CTR, average position

Conclusion:
No future-window or product-flag leakage was used in the baseline scoring rule.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.